**Step 1: Idempotency check** – reran the same stream with no new files in the source.
Row count before and after: 25,820 -> 25,820. Confirms checkpoint prevents reprocessing
already-ingested files.

In [0]:
%sql
SELECT COUNT(*) AS row_count FROM dbr_dev_ua5816bd.roksolana_shendiu770_bronze.petroleum_consumption_bronze

In [0]:
from pyspark.sql.functions import col

CATALOG = "dbr_dev_ua5816bd"
SCHEMA_LANDING = "roksolana_shendiu770"
SCHEMA_BRONZE = "roksolana_shendiu770_bronze"

SOURCE_PATH = f"/Volumes/{CATALOG}/{SCHEMA_LANDING}/bronze_landing/petroleum_consumption"
SCHEMA_LOCATION = f"/Volumes/{CATALOG}/{SCHEMA_LANDING}/bronze_landing/_schemas/petroleum_consumption"
CHECKPOINT_LOCATION = f"/Volumes/{CATALOG}/{SCHEMA_LANDING}/bronze_landing/_checkpoints/petroleum_consumption"
TARGET_TABLE = f"{CATALOG}.{SCHEMA_BRONZE}.petroleum_consumption_bronze"

raw_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", SCHEMA_LOCATION)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaEvolutionMode", "addNewColumnsWithTypeWidening")
    .option("cloudFiles.maxFilesPerTrigger", 100)
    .load(SOURCE_PATH)
    .select("*", col("_metadata.file_path").alias("source_file_path"))
)

query = (
    raw_stream.writeStream
    .option("checkpointLocation", CHECKPOINT_LOCATION)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(TARGET_TABLE)
)

query.awaitTermination()

In [0]:
%sql
SELECT COUNT(*) AS row_count FROM dbr_dev_ua5816bd.roksolana_shendiu770_bronze.petroleum_consumption_bronze

**Step 2: Checkpoint loss risk** – deleted the checkpoint directory and reran the
stream twice. Each time, Auto Loader lost track of processed files and reprocessed
the entire landing zone via full directory listing, compounding duplicates each time
(25,820 -> 77,612 total rows). Demonstrates that checkpoint loss is not a one-time
risk – repeated loss multiplies duplication.

In [0]:
dbutils.fs.rm(CHECKPOINT_LOCATION, recurse=True)
print("Checkpoint deleted.")

In [0]:
%sql
SELECT COUNT(*) AS row_count FROM dbr_dev_ua5816bd.roksolana_shendiu770_bronze.petroleum_consumption_bronze

**Step 3: Safe reload** – cleared checkpoint, schema location, AND target table
together, then reran the stream. Result: 25,896 rows – clean, no accumulated
duplicates from the previous checkpoint-loss experiments. Correct way to "start
fresh": all three (checkpoint, schema location, table) must be reset together,
not just the checkpoint alone.

In [0]:

dbutils.fs.rm(CHECKPOINT_LOCATION, recurse=True)
dbutils.fs.rm(SCHEMA_LOCATION, recurse=True)
spark.sql(f"DROP TABLE IF EXISTS {TARGET_TABLE}")
print("Checkpoint, schema location, and table cleared for safe reload.")

In [0]:
%sql
SELECT COUNT(*) AS row_count FROM dbr_dev_ua5816bd.roksolana_shendiu770_bronze.petroleum_consumption_bronze